# 실험 결1: 피처 구성 / 결측치 전략 비교

**질문**: 결측치를 어떻게 대치하고, 어떤 피처를 쓰느냐가 모델 성능에 통계적으로 유의미한 차이를 만드는가?

**고정 조건 (이번 실험에서는 바꾸지 않음)**
- 모델: `LogisticRegression` 고정 — 모델 종류 비교는 다음 실험(`02_model_type.ipynb`)에서 다룬다.
- **train/test 분할을 실험 시작 시 한 번만 하고, 모든 변형이 동일한 test set을 공유한다.** McNemar's test는 같은 표본에 대한 두 모델의 예측을 쌍으로 비교하는 것이라, test set이 다르면 비교 자체가 성립하지 않는다.
- 그래서 이번 실험은 전부 "결측치 대치" 계열만 다룬다. "결측 행 제거"는 test set의 행 구성 자체가 바뀌어 버려서 이 방식으로는 공정 비교가 안 된다 — 필요하면 별도 실험에서 complete-case subset 기준으로 다뤄야 한다.

**비교할 변형**
- A (baseline): 전체 피처, median/mode 대치, `education` 제외 (`education-num`과 중복 정보)
- B: A + 결측 여부 indicator 컬럼 추가
- C: `education-num`(수치형) 대신 `education`(범주형, 원-핫) 사용
- D: `fnlwgt` 제외 (표본 가중치일 뿐 개인 특성이 아님)

각 셀을 순서대로 하나씩 실행하면서 로그와 McNemar 결과를 확인하면 된다. 새 변형을 추가하고 싶으면 아래 패턴을 복사해서 셀을 하나 더 만들면 됨.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import common
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

EXPERIMENT_GROUP = "feature_missing_strategy"

df = common.load_raw()

# 모든 변형이 공유하는 단일 test set. random_state 고정 필수.
df_train, df_test = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df[common.TARGET_COL]
)
print("train:", df_train.shape, "test:", df_test.shape)

# 이후 각 변형 셀에서 y_test와 baseline 예측을 참조하므로 여기 담아둔다.
baseline_pred = None

train: (26048, 15) test: (6513, 15)


## 변형 A (baseline) — 전체 피처, median/mode 대치, `education` 제외

In [2]:
RUN_NAME = "A_baseline_median_mode"
DESCRIPTION = "median/mode 대치, education 제외(education-num과 중복이라 제외)"

numeric_cols = common.ALL_NUMERIC_COLS
categorical_cols = [c for c in common.ALL_CATEGORICAL_COLS if c != "education"]
feature_cols = numeric_cols + categorical_cols

X_train, y_train = common.make_xy(df_train, feature_cols)
X_test, y_test = common.make_xy(df_test, feature_cols)

pipeline_a = common.make_pipeline(
    numeric_cols, categorical_cols, LogisticRegression(max_iter=1000),
    numeric_impute_strategy="median", categorical_impute_strategy="most_frequent",
)
pipeline_a.fit(X_train, y_train)
result_a = common.evaluate(pipeline_a, X_test, y_test)
print(result_a["metrics"])

common.log_result(RUN_NAME, EXPERIMENT_GROUP, DESCRIPTION, result_a["metrics"])

# 다음 변형들이 비교할 기준으로 저장
baseline_pred = result_a["y_pred"]
baseline_run_name = RUN_NAME

{'accuracy': 0.8549055734684478, 'precision': 0.7358062074186222, 'recall': 0.6198979591836735, 'f1': 0.6728971962616822, 'roc_auc': 0.907816723757248}
[logged] A_baseline_median_mode -> /Users/hyo/skala/sk-log/data-project/team_project/results/experiment_log.csv


## 변형 B — A + 결측 여부 indicator 컬럼 추가

In [3]:
RUN_NAME = "B_missing_indicator"
DESCRIPTION = "A와 동일 + SimpleImputer(add_indicator=True)로 결측 여부 플래그 추가"

numeric_cols = common.ALL_NUMERIC_COLS
categorical_cols = [c for c in common.ALL_CATEGORICAL_COLS if c != "education"]
feature_cols = numeric_cols + categorical_cols

X_train, y_train = common.make_xy(df_train, feature_cols)
X_test, y_test = common.make_xy(df_test, feature_cols)

pipeline_b = common.make_pipeline(
    numeric_cols, categorical_cols, LogisticRegression(max_iter=1000),
    numeric_impute_strategy="median", categorical_impute_strategy="most_frequent",
    add_missing_indicator=True,
)
pipeline_b.fit(X_train, y_train)
result_b = common.evaluate(pipeline_b, X_test, y_test)
print(result_b["metrics"])

cmp_b = common.compare_models(y_test, baseline_pred, result_b["y_pred"], baseline_run_name, RUN_NAME)
print(cmp_b["interpretation"])

common.log_result(RUN_NAME, EXPERIMENT_GROUP, DESCRIPTION, result_b["metrics"],
                   compared_to=baseline_run_name, mcnemar_result=cmp_b)

{'accuracy': 0.8545984953170582, 'precision': 0.7361216730038023, 'recall': 0.6173469387755102, 'f1': 0.6715227193895248, 'roc_auc': 0.909911578382617}
p=0.9131 >= 0.05: 'A_baseline_median_mode'와 'B_missing_indicator'의 오류 패턴 차이가 통계적으로 유의하지 않음 (우연일 수 있음)
[logged] B_missing_indicator -> /Users/hyo/skala/sk-log/data-project/team_project/results/experiment_log.csv


## 변형 C — `education-num` 대신 `education`(범주형, 원-핫) 사용

In [4]:
RUN_NAME = "C_education_categorical"
DESCRIPTION = "education-num 제거하고 education(범주형, one-hot) 사용"

numeric_cols = [c for c in common.ALL_NUMERIC_COLS if c != "education-num"]
categorical_cols = common.ALL_CATEGORICAL_COLS  # education 포함
feature_cols = numeric_cols + categorical_cols

X_train, y_train = common.make_xy(df_train, feature_cols)
X_test, y_test = common.make_xy(df_test, feature_cols)

pipeline_c = common.make_pipeline(
    numeric_cols, categorical_cols, LogisticRegression(max_iter=1000),
    numeric_impute_strategy="median", categorical_impute_strategy="most_frequent",
)
pipeline_c.fit(X_train, y_train)
result_c = common.evaluate(pipeline_c, X_test, y_test)
print(result_c["metrics"])

cmp_c = common.compare_models(y_test, baseline_pred, result_c["y_pred"], baseline_run_name, RUN_NAME)
print(cmp_c["interpretation"])

common.log_result(RUN_NAME, EXPERIMENT_GROUP, DESCRIPTION, result_c["metrics"],
                   compared_to=baseline_run_name, mcnemar_result=cmp_c)

{'accuracy': 0.855366190695532, 'precision': 0.7400306748466258, 'recall': 0.6154336734693877, 'f1': 0.6720055710306406, 'roc_auc': 0.9077813860630197}
p=0.7874 >= 0.05: 'A_baseline_median_mode'와 'C_education_categorical'의 오류 패턴 차이가 통계적으로 유의하지 않음 (우연일 수 있음)
[logged] C_education_categorical -> /Users/hyo/skala/sk-log/data-project/team_project/results/experiment_log.csv


## 변형 D — `fnlwgt` 제외 (표본 가중치일 뿐 개인 특성이 아님)

In [5]:
RUN_NAME = "D_drop_fnlwgt"
DESCRIPTION = "A와 동일 + fnlwgt 제외"

numeric_cols = [c for c in common.ALL_NUMERIC_COLS if c != "fnlwgt"]
categorical_cols = [c for c in common.ALL_CATEGORICAL_COLS if c != "education"]
feature_cols = numeric_cols + categorical_cols

X_train, y_train = common.make_xy(df_train, feature_cols)
X_test, y_test = common.make_xy(df_test, feature_cols)

pipeline_d = common.make_pipeline(
    numeric_cols, categorical_cols, LogisticRegression(max_iter=1000),
    numeric_impute_strategy="median", categorical_impute_strategy="most_frequent",
)
pipeline_d.fit(X_train, y_train)
result_d = common.evaluate(pipeline_d, X_test, y_test)
print(result_d["metrics"])

cmp_d = common.compare_models(y_test, baseline_pred, result_d["y_pred"], baseline_run_name, RUN_NAME)
print(cmp_d["interpretation"])

common.log_result(RUN_NAME, EXPERIMENT_GROUP, DESCRIPTION, result_d["metrics"],
                   compared_to=baseline_run_name, mcnemar_result=cmp_d)

{'accuracy': 0.8550591125441425, 'precision': 0.7381679389312977, 'recall': 0.6167091836734694, 'f1': 0.6719944405837387, 'roc_auc': 0.9075654908070407}
p=1 >= 0.05: 'A_baseline_median_mode'와 'D_drop_fnlwgt'의 오류 패턴 차이가 통계적으로 유의하지 않음 (우연일 수 있음)
[logged] D_drop_fnlwgt -> /Users/hyo/skala/sk-log/data-project/team_project/results/experiment_log.csv


## 변형 E — 결측을 최빈값으로 채우지 않고 별도 범주 `"Missing"`으로 유지

workclass 결측은 occupation 결측과 100% 겹치고, 결측 있는 행의 `>50K` 비율(13.9%)이 없는 행(24.9%)의 절반 수준이다.
즉 결측이 무작위가 아니라 그 자체로 신호다. 최빈값으로 덮어쓰지 않고 `"Missing"`이라는 범주를 그대로 원-핫에 남겨서
모델이 이 신호를 직접 쓸 수 있게 한다. 행 수·test set 구성은 baseline과 동일하므로 McNemar로 바로 비교 가능.

In [6]:
RUN_NAME = "E_missing_as_category"
DESCRIPTION = "범주형 결측을 최빈값 대신 'Missing' 별도 범주로 유지 (workclass/occupation 결측이 income과 연관돼 있어 신호로 보존)"

numeric_cols = common.ALL_NUMERIC_COLS
categorical_cols = [c for c in common.ALL_CATEGORICAL_COLS if c != "education"]
feature_cols = numeric_cols + categorical_cols

X_train, y_train = common.make_xy(df_train, feature_cols)
X_test, y_test = common.make_xy(df_test, feature_cols)

pipeline_e = common.make_pipeline(
    numeric_cols, categorical_cols, LogisticRegression(max_iter=1000),
    numeric_impute_strategy="median",
    categorical_impute_strategy="constant", categorical_fill_value="Missing",
)
pipeline_e.fit(X_train, y_train)
result_e = common.evaluate(pipeline_e, X_test, y_test)
print(result_e["metrics"])

cmp_e = common.compare_models(y_test, baseline_pred, result_e["y_pred"], baseline_run_name, RUN_NAME)
print(cmp_e["interpretation"])

common.log_result(RUN_NAME, EXPERIMENT_GROUP, DESCRIPTION, result_e["metrics"],
                   compared_to=baseline_run_name, mcnemar_result=cmp_e)

{'accuracy': 0.8542914171656687, 'precision': 0.7353612167300381, 'recall': 0.6167091836734694, 'f1': 0.6708289975719737, 'roc_auc': 0.9098775303852583}
p=0.7434 >= 0.05: 'A_baseline_median_mode'와 'E_missing_as_category'의 오류 패턴 차이가 통계적으로 유의하지 않음 (우연일 수 있음)
[logged] E_missing_as_category -> /Users/hyo/skala/sk-log/data-project/team_project/results/experiment_log.csv


## 새 변형을 추가하려면

위 셀 하나를 복사해서: `RUN_NAME`/`DESCRIPTION` 바꾸고 → `numeric_cols`/`categorical_cols`/impute 전략만 원하는 대로 수정하고 → 나머지(`make_xy` ~ `log_result`)는 그대로 두면 된다. `baseline_pred`, `baseline_run_name`은 맨 위 변형 A 셀에서 이미 정의돼 있으니 계속 재사용.

## 지금까지의 결과 한눈에 보기

In [7]:
common.show_log(experiment_group=EXPERIMENT_GROUP)

,timestamp,experiment_group,run_name,description,accuracy,precision,recall,f1,roc_auc,compared_to,mcnemar_p_value,mcnemar_significant,mcnemar_interpretation
0,2026-08-07T16:21:23,feature_missing_strategy,A_baseline_median_mode,"median/mode 대치, education 제외(education-num과 중복...",0.854906,0.735806,0.619898,0.672897,0.907817,NaN,NaN,NaN,NaN
4,2026-08-07T16:28:01,feature_missing_strategy,A_baseline_median_mode,"median/mode 대치, education 제외(education-num과 중복...",0.854906,0.735806,0.619898,0.672897,0.907817,NaN,NaN,NaN,NaN
2,2026-08-07T16:21:24,feature_missing_strategy,C_education_categorical,"education-num 제거하고 education(범주형, one-hot) 사용",0.855366,0.740031,0.615434,0.672006,0.907781,A_baseline_median_mode,0.787406,False,p=0.7874 >= 0.05: 'A_baseline_median_mode'와 'C...
6,2026-08-07T16:28:02,feature_missing_strategy,C_education_categorical,"education-num 제거하고 education(범주형, one-hot) 사용",0.855366,0.740031,0.615434,0.672006,0.907781,A_baseline_median_mode,0.787406,False,p=0.7874 >= 0.05: 'A_baseline_median_mode'와 'C...
3,2026-08-07T16:21:24,feature_missing_strategy,D_drop_fnlwgt,A와 동일 + fnlwgt 제외,0.855059,0.738168,0.616709,0.671994,0.907565,A_baseline_median_mode,1.000000,False,p=1 >= 0.05: 'A_baseline_median_mode'와 'D_drop...
7,2026-08-07T16:28:02,feature_missing_strategy,D_drop_fnlwgt,A와 동일 + fnlwgt 제외,0.855059,0.738168,0.616709,0.671994,0.907565,A_baseline_median_mode,1.000000,False,p=1 >= 0.05: 'A_baseline_median_mode'와 'D_drop...
1,2026-08-07T16:21:23,feature_missing_strategy,B_missing_indicator,A와 동일 + SimpleImputer(add_indicator=True)로 결측 ...,0.854598,0.736122,0.617347,0.671523,0.909912,A_baseline_median_mode,0.913116,False,p=0.9131 >= 0.05: 'A_baseline_median_mode'와 'B...
5,2026-08-07T16:28:01,feature_missing_strategy,B_missing_indicator,A와 동일 + SimpleImputer(add_indicator=True)로 결측 ...,0.854598,0.736122,0.617347,0.671523,0.909912,A_baseline_median_mode,0.913116,False,p=0.9131 >= 0.05: 'A_baseline_median_mode'와 'B...
8,2026-08-07T16:28:02,feature_missing_strategy,E_missing_as_category,범주형 결측을 최빈값 대신 'Missing' 별도 범주로 유지 (workclass/...,0.854291,0.735361,0.616709,0.670829,0.909878,A_baseline_median_mode,0.743421,False,p=0.7434 >= 0.05: 'A_baseline_median_mode'와 'E...
